# DGL Dataset Preprocessing for QuVINE

This notebook helps you:

1. Download the DGL citation network dataset
2. Convert it into a clean undirected NetworkX graph
3. Extract repeated **connected induced subgraphs** of approximately **2k, 5k, 10k nodes**
4. Use a degree-aware sampling heuristic to better preserve the original degree distribution
5. Save sampled subgraphs as edge lists and metadata for downstream QuVINE processing

## Dataset

- **DGL datasets**: Citation network from arXiv CS papers
- **Nodes**: 169,343 papers
- **Edges**: 1,166,243 citations
- **Features**: 128-dimensional node features
- **Labels**: 40 subject areas
- **Task**: Node classification

## Design choices

- **Loader**: OGB `DGL dataset classes`
- **Sampling goal**: connected, induced, degree-aware, repeated multiple times (30 replicas per size)
- **Output format**: CSV edgelists + JSON metadata + NumPy arrays for features/labels/masks

## Notes

- The degree-preservation step here is heuristic, not an exact constrained optimization.
- Subgraph sampling preserves train/val/test splits for sampled nodes.

## 1. Environment and imports

In [ ]:
# Uncomment to install required packages
# %pip install ogb networkx numpy pandas matplotlib scipy tqdm

In [ ]:
from __future__ import annotations

import json
from collections import deque
from pathlib import Path
from typing import Dict, Iterable, List

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from ogb.nodeproppred import DGL dataset classes
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
DATASET_NAME = "DGL datasets"

# Output location
OUTPUT_DIR = Path("/dccstor/cgq4hls/Q/dgl/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sampling settings
TARGET_SIZES = [2000, 5000, 10000]
REPEATS = 30
BASE_SEED = 42

# Degree-aware sampling heuristic settings
RESTART_PROB = 0.15
FRONTIER_WIDTH = 256
DEGREE_WEIGHT_POWER = 0.5
LOCAL_BFS_EXPANSION = 8

print("Output dir:", OUTPUT_DIR.resolve())

## 3. Load DGL Dataset

In [ ]:
# Load dataset
print(f"Loading {DATASET_NAME}...")
dataset = DGL dataset classes(name=DATASET_NAME)

# Extract graph and splits
graph, labels = dataset[0]
split_idx = dataset.get_idx_split()

# Extract components
edge_index = graph['edge_index']  # Shape: [2, num_edges]
node_features = graph['node_feat']  # Shape: [num_nodes, num_features]
node_labels = labels.squeeze()  # Shape: [num_nodes]

train_idx = split_idx['train']
val_idx = split_idx['valid']
test_idx = split_idx['test']

n_nodes = node_features.shape[0]
n_features = node_features.shape[1]
n_classes = int(node_labels.max()) + 1

print(f"Nodes: {n_nodes:,}")
print(f"Edges: {edge_index.shape[1]:,}")
print(f"Features: {n_features}")
print(f"Classes: {n_classes}")
print(f"Train: {len(train_idx):,}")
print(f"Val: {len(val_idx):,}")
print(f"Test: {len(test_idx):,}")

## 4. Convert to NetworkX and Create Masks

In [ ]:
# Convert edge_index to edge list
edges = edge_index.T  # Shape: [num_edges, 2]

# Create train/val/test masks (single split)
train_mask = np.zeros((n_nodes, 1), dtype=bool)
val_mask = np.zeros((n_nodes, 1), dtype=bool)
test_mask = np.zeros((n_nodes, 1), dtype=bool)

train_mask[train_idx, 0] = True
val_mask[val_idx, 0] = True
test_mask[test_idx, 0] = True

print(f"Train mask shape: {train_mask.shape}, sum: {train_mask.sum()}")
print(f"Val mask shape: {val_mask.shape}, sum: {val_mask.sum()}")
print(f"Test mask shape: {test_mask.shape}, sum: {test_mask.sum()}")

In [ ]:
# Build NetworkX graph
def build_graph_from_edges(edges: np.ndarray, n_nodes: int) -> nx.Graph:
    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    G.add_edges_from((int(u), int(v)) for u, v in edges if int(u) != int(v))
    return G

print("Building NetworkX graph...")
G_full = build_graph_from_edges(edges, n_nodes)
print(f"Graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges")
print(f"Connected: {nx.is_connected(G_full)}")
print(f"Connected components: {nx.number_connected_components(G_full)}")

## 5. Graph conversion and sampling helpers

In [ ]:
def graph_degree_array(G: nx.Graph) -> np.ndarray:
    if G.number_of_nodes() == 0:
        return np.array([], dtype=float)
    return np.array([d for _, d in G.degree()], dtype=float)


def materialize_undirected_simple_graph(G: nx.Graph) -> nx.Graph:
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from(G.edges(data=True))
    H.remove_edges_from(nx.selfloop_edges(H))
    return H


def induce_subgraph_by_nodes(G: nx.Graph, nodes: Iterable) -> nx.Graph:
    node_set = set(nodes)
    H = nx.Graph()
    H.add_nodes_from((n, G.nodes[n]) for n in node_set)
    H.add_edges_from((u, v, d) for u, v, d in G.edges(data=True) if u in node_set and v in node_set)
    return H


def keep_largest_connected_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    lcc_nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(lcc_nodes).copy()


def weighted_choice_without_replacement(items, weights, k, rng):
    items = list(items)
    weights = np.asarray(weights, dtype=float)
    if len(items) == 0:
        return []
    if np.all(weights <= 0):
        weights = np.ones(len(items), dtype=float)
    weights = np.maximum(weights, 1e-12)
    probs = weights / weights.sum()
    k_eff = min(k, len(items))
    idx = rng.choice(len(items), size=k_eff, replace=False, p=probs)
    return [items[i] for i in idx]


def sample_anchor_node(G: nx.Graph, rng: np.random.Generator, power: float = 0.5):
    nodes = list(G.nodes())
    deg = np.array([G.degree(n) for n in nodes], dtype=float)
    weights = np.power(np.maximum(deg, 1.0), power)
    weights = weights / weights.sum()
    return nodes[int(rng.choice(len(nodes), p=weights))]


def sample_degree_targets(G: nx.Graph, sample_size: int, rng: np.random.Generator) -> np.ndarray:
    deg = graph_degree_array(G)
    if len(deg) == 0:
        return np.zeros(sample_size)
    return rng.choice(deg, size=sample_size, replace=True)


def summarize_graph(G: nx.Graph) -> Dict[str, float]:
    deg = graph_degree_array(G)
    return {
        "num_nodes": int(G.number_of_nodes()),
        "num_edges": int(G.number_of_edges()),
        "avg_degree": float(deg.mean()) if len(deg) else 0.0,
        "median_degree": float(np.median(deg)) if len(deg) else 0.0,
        "max_degree": float(deg.max()) if len(deg) else 0.0,
        "num_connected_components": int(nx.number_connected_components(G)),
    }


def degree_histogram_distance(G_ref: nx.Graph, G_sub: nx.Graph, bins: int = 30) -> float:
    deg_ref = graph_degree_array(G_ref)
    deg_sub = graph_degree_array(G_sub)
    if len(deg_ref) == 0 or len(deg_sub) == 0:
        return float("inf")
    upper = max(float(deg_ref.max()), float(deg_sub.max()), 1.0)
    bin_edges = np.linspace(0.0, upper, bins + 1)
    h_ref, _ = np.histogram(deg_ref, bins=bin_edges, density=True)
    h_sub, _ = np.histogram(deg_sub, bins=bin_edges, density=True)
    return float(np.abs(h_ref - h_sub).sum())


full_summary = summarize_graph(G_full)
print("Full graph summary:")
print(full_summary)

## 6. Connected induced subgraph sampling

### Heuristic

The sampler below uses a degree-aware frontier expansion strategy:

1. Pick an anchor node with probability weighted by degree
2. Grow a connected set using a frontier
3. Score candidate frontier nodes using:
   - internal connectivity to the current sample
   - closeness of node degree to a degree target drawn from the full graph
   - a small exploration / restart mechanism
4. Materialize the **induced** subgraph on the selected nodes
5. Keep the largest connected component if needed

This is not exact degree-sequence matching, but it works well as a practical preprocessing heuristic.

In [ ]:
def connected_degree_aware_subgraph(
    G: nx.Graph,
    target_size: int,
    rng: np.random.Generator,
    restart_prob: float = 0.15,
    degree_weight_power: float = 0.5,
    frontier_width: int = 256,
    local_bfs_expansion: int = 8,
) -> nx.Graph:
    if target_size >= G.number_of_nodes():
        return materialize_undirected_simple_graph(G)

    anchor = sample_anchor_node(G, rng, power=degree_weight_power)
    selected = [anchor]
    selected_set = {anchor}
    frontier = set(G.adj[anchor].keys())
    degree_targets = sample_degree_targets(G, target_size, rng)
    target_ptr = 0

    while len(selected) < target_size:
        if not frontier or rng.random() < restart_prob:
            seed_from_selected = selected[int(rng.integers(0, len(selected)))]
            local_frontier = deque([seed_from_selected])
            steps = 0
            while local_frontier and steps < local_bfs_expansion:
                u = local_frontier.popleft()
                nbrs = list(G.adj[u].keys())
                rng.shuffle(nbrs)
                for v in nbrs:
                    if v not in selected_set:
                        frontier.add(v)
                        local_frontier.append(v)
                steps += 1

        if not frontier:
            remaining = list(set(G.nodes()) - selected_set)
            if not remaining:
                break
            candidate = remaining[int(rng.integers(0, len(remaining)))]
            frontier.add(candidate)

        frontier_list = list(frontier)
        if len(frontier_list) > frontier_width:
            frontier_list = weighted_choice_without_replacement(
                frontier_list,
                [max(G.degree(n), 1) for n in frontier_list],
                frontier_width,
                rng,
            )

        target_degree = degree_targets[min(target_ptr, len(degree_targets) - 1)]
        scores = []
        for node in frontier_list:
            deg = G.degree(node)
            internal_links = sum((nbr in selected_set) for nbr in G.adj[node].keys())
            degree_match = 1.0 / (1.0 + abs(deg - target_degree))
            score = 2.0 * internal_links + degree_match + 0.25 * np.log1p(deg)
            scores.append(score)

        scores = np.asarray(scores, dtype=float)
        if np.all(scores <= 0):
            scores = np.ones_like(scores)
        probs = scores / scores.sum()
        chosen = frontier_list[int(rng.choice(len(frontier_list), p=probs))]

        selected.append(chosen)
        selected_set.add(chosen)
        frontier.discard(chosen)
        frontier.update(v for v in G.adj[chosen].keys() if v not in selected_set)
        target_ptr += 1

    H = induce_subgraph_by_nodes(G, selected_set)
    H = materialize_undirected_simple_graph(H)
    H = keep_largest_connected_component(H)

    while H.number_of_nodes() < target_size and H.number_of_nodes() < G.number_of_nodes():
        current_nodes = set(H.nodes())
        boundary = set()
        for u in current_nodes:
            boundary.update(v for v in G.adj[u].keys() if v not in current_nodes)
        if not boundary:
            break
        boundary = list(boundary)
        rng.shuffle(boundary)
        needed = min(target_size - H.number_of_nodes(), len(boundary))
        current_nodes.update(boundary[:needed])
        H = keep_largest_connected_component(induce_subgraph_by_nodes(G, current_nodes))

    return H

## 7. Export helpers

In [ ]:
def save_graph_edgelist_csv(edges: np.ndarray, path: Path):
    df = pd.DataFrame(edges, columns=["node1", "node2"])
    df.to_csv(path, index=False)


def save_node_index_csv(exported_node_ids: np.ndarray, original_node_ids: np.ndarray, path: Path):
    df = pd.DataFrame({
        "export_node_id": exported_node_ids.astype(int),
        "original_node_id": original_node_ids.astype(int),
    })
    df.to_csv(path, index=False)


def make_export_bundle_arrays(
    original_edges: np.ndarray,
    node_features: np.ndarray,
    node_labels: np.ndarray,
    train_masks: np.ndarray,
    val_masks: np.ndarray,
    test_masks: np.ndarray,
    selected_nodes: np.ndarray,
) -> Dict[str, np.ndarray]:
    selected_nodes = np.asarray(selected_nodes, dtype=np.int64)
    selected_set = set(selected_nodes.tolist())
    old_to_new = {int(old): int(new) for new, old in enumerate(selected_nodes.tolist())}

    edge_mask = np.array(
        [(int(u) in selected_set) and (int(v) in selected_set) for u, v in original_edges],
        dtype=bool,
    )
    sub_edges = original_edges[edge_mask]
    if len(sub_edges) == 0:
        remapped_edges = np.empty((0, 2), dtype=np.int64)
    else:
        remapped_edges = np.array([[old_to_new[int(u)], old_to_new[int(v)]] for u, v in sub_edges], dtype=np.int64)

    return {
        "selected_original_nodes": selected_nodes,
        "exported_node_ids": np.arange(len(selected_nodes), dtype=np.int64),
        "edges": remapped_edges,
        "node_features": node_features[selected_nodes],
        "node_labels": node_labels[selected_nodes],
        "train_masks": train_masks[selected_nodes],
        "val_masks": val_masks[selected_nodes],
        "test_masks": test_masks[selected_nodes],
    }


def write_bundle(
    bundle_dir: Path,
    stem: str,
    arrays: Dict[str, np.ndarray],
    metadata: Dict,
):
    bundle_dir.mkdir(parents=True, exist_ok=True)
    csv_path = bundle_dir / f"{stem}.csv"
    json_path = bundle_dir / f"{stem}.json"
    labels_path = bundle_dir / f"{stem}_node_labels.npy"
    features_path = bundle_dir / f"{stem}_node_features.npy"
    train_masks_path = bundle_dir / f"{stem}_train_masks.npy"
    val_masks_path = bundle_dir / f"{stem}_val_masks.npy"
    test_masks_path = bundle_dir / f"{stem}_test_masks.npy"
    node_index_path = bundle_dir / f"{stem}_node_index.csv"

    save_graph_edgelist_csv(arrays["edges"], csv_path)
    np.save(labels_path, arrays["node_labels"])
    np.save(features_path, arrays["node_features"])
    np.save(train_masks_path, arrays["train_masks"])
    np.save(val_masks_path, arrays["val_masks"])
    np.save(test_masks_path, arrays["test_masks"])
    save_node_index_csv(arrays["exported_node_ids"], arrays["selected_original_nodes"], node_index_path)

    metadata = dict(metadata)
    metadata.update({
        "csv_path": str(csv_path),
        "labels_path": str(labels_path),
        "features_path": str(features_path),
        "train_masks_path": str(train_masks_path),
        "val_masks_path": str(val_masks_path),
        "test_masks_path": str(test_masks_path),
        "node_index_path": str(node_index_path),
    })

    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "csv": csv_path,
        "json": json_path,
        "labels": labels_path,
        "features": features_path,
        "train_masks": train_masks_path,
        "val_masks": val_masks_path,
        "test_masks": test_masks_path,
        "node_index": node_index_path,
    }


def split_counts(mask_array: np.ndarray) -> List[int]:
    return [int(mask_array[:, i].sum()) for i in range(mask_array.shape[1])]

## 8. Sample and export subgraphs

In [ ]:
all_samples = []

for target_size in TARGET_SIZES:
    print(f"\n{'='*60}")
    print(f"Sampling {REPEATS} subgraphs of target size {target_size}")
    print(f"{'='*60}")
    
    for repeat_idx in tqdm(range(REPEATS), desc=f"Size {target_size}"):
        seed = BASE_SEED + 1000 * repeat_idx + target_size
        rng = np.random.default_rng(seed)
        
        # Sample subgraph
        H = connected_degree_aware_subgraph(
            G_full,
            target_size=target_size,
            rng=rng,
            restart_prob=RESTART_PROB,
            degree_weight_power=DEGREE_WEIGHT_POWER,
            frontier_width=FRONTIER_WIDTH,
            local_bfs_expansion=LOCAL_BFS_EXPANSION,
        )

        selected_nodes = np.array(sorted(H.nodes()), dtype=np.int64)
        
        # Create export arrays
        arrays = make_export_bundle_arrays(
            original_edges=edges,
            node_features=node_features,
            node_labels=node_labels,
            train_masks=train_mask,
            val_masks=val_mask,
            test_masks=test_mask,
            selected_nodes=selected_nodes,
        )

        # Compute quality metrics
        summary = summarize_graph(H)
        degree_hist_l1 = degree_histogram_distance(G_full, H)
        
        # Create metadata
        stem = f"dgl_n{target_size}_rep{repeat_idx:02d}"
        metadata = {
            "dataset_family": "ogb",
            "dataset_name": "DGL datasets",
            "network_id": stem,
            "type": "ogb_subsample",
            "graph_name": stem,
            "n_nodes": int(arrays["node_features"].shape[0]),
            "n_edges": int(arrays["edges"].shape[0]),
            "n_features": int(arrays["node_features"].shape[1]),
            "n_classes": int(np.unique(arrays["node_labels"]).size),
            "n_splits": int(arrays["train_masks"].shape[1]),
            "train_counts_by_split": split_counts(arrays["train_masks"]),
            "val_counts_by_split": split_counts(arrays["val_masks"]),
            "test_counts_by_split": split_counts(arrays["test_masks"]),
            "original_num_nodes": n_nodes,
            "sampling": {
                "target_size": int(target_size),
                "effective_target_size": int(arrays["node_features"].shape[0]),
                "repeat_idx": int(repeat_idx),
                "seed": int(seed),
                "degree_hist_l1": float(degree_hist_l1),
                "restart_prob": RESTART_PROB,
                "degree_weight_power": DEGREE_WEIGHT_POWER,
                "frontier_width": FRONTIER_WIDTH,
                "local_bfs_expansion": LOCAL_BFS_EXPANSION,
            },
        }
        metadata.update(summary)

        # Write bundle
        subdir = OUTPUT_DIR / f"n{target_size}"
        paths = write_bundle(subdir, stem, arrays, metadata)
        
        # Record sample info
        all_samples.append({
            "target_size": int(target_size),
            "effective_target_size": int(arrays["node_features"].shape[0]),
            "repeat_idx": int(repeat_idx),
            "seed": int(seed),
            "degree_hist_l1": float(degree_hist_l1),
            "num_nodes": summary["num_nodes"],
            "num_edges": summary["num_edges"],
            "avg_degree": summary["avg_degree"],
            "median_degree": summary["median_degree"],
            "max_degree": summary["max_degree"],
            "csv_path": str(paths["csv"]),
            "json_path": str(paths["json"]),
        })

print(f"\n{'='*60}")
print("Sampling complete!")
print(f"{'='*60}")

## 9. Results summary

In [ ]:
results_df = pd.DataFrame(all_samples)
print("\nAll samples:")
print(results_df.head(10))

# Save manifest
manifest_path = OUTPUT_DIR / "manifest.csv"
results_df.to_csv(manifest_path, index=False)
print(f"\nSaved manifest: {manifest_path}")

## 10. Quality diagnostics

In [ ]:
# Degree histogram distance statistics
quality_summary = results_df.groupby("target_size")["degree_hist_l1"].agg(["mean", "std", "min", "max"]).reset_index()
print("\nDegree histogram distance by target size:")
print(quality_summary)

# Size statistics
size_summary = results_df.groupby("target_size").agg({
    "effective_target_size": ["mean", "std"],
    "num_edges": ["mean", "std"],
    "avg_degree": ["mean", "std"],
}).reset_index()
print("\nSize statistics by target:")
print(size_summary)

## 11. Visualize degree distribution comparison

In [ ]:
# Pick best sample from each target size (lowest degree_hist_l1)
best_samples = results_df.sort_values("degree_hist_l1").groupby("target_size").first().reset_index()

# Load and plot degree distributions
plt.figure(figsize=(10, 6))

# Full graph
deg_full = graph_degree_array(G_full)
deg_full = deg_full[deg_full > 0]
xs_full = np.sort(np.unique(deg_full))
ccdf_full = np.array([(deg_full >= x).mean() for x in xs_full])
plt.step(xs_full, ccdf_full, where="post", label="Full graph", linewidth=2)

# Best samples
for _, row in best_samples.iterrows():
    json_path = Path(row["json_path"])
    csv_path = json_path.parent / f"{json_path.stem.replace('.json', '')}.csv"
    
    # Load sample graph
    df = pd.read_csv(csv_path)
    G_sample = nx.Graph()
    G_sample.add_edges_from(zip(df["node1"], df["node2"]))
    
    deg_sample = graph_degree_array(G_sample)
    deg_sample = deg_sample[deg_sample > 0]
    xs_sample = np.sort(np.unique(deg_sample))
    ccdf_sample = np.array([(deg_sample >= x).mean() for x in xs_sample])
    plt.step(xs_sample, ccdf_sample, where="post", 
             label=f"Sample n={row['target_size']} (L1={row['degree_hist_l1']:.3f})", alpha=0.7)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Degree")
plt.ylabel("CCDF")
plt.title("Degree Distribution: Full Graph vs Samples")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Complete!

The DGL dataset has been preprocessed and exported in QuVINE-compatible format.

### Output Structure:

```
processed/
├── n2000/
│   ├── dgl_n2000_rep00.csv
│   ├── dgl_n2000_rep00.json
│   ├── dgl_n2000_rep00_node_labels.npy
│   ├── dgl_n2000_rep00_node_features.npy
│   ├── dgl_n2000_rep00_train_masks.npy
│   ├── dgl_n2000_rep00_val_masks.npy
│   ├── dgl_n2000_rep00_test_masks.npy
│   ├── dgl_n2000_rep00_node_index.csv
│   └── ... (30 repeats)
├── n5000/
├── n10000/
└── manifest.csv
```